In [1]:
import pandas as pd 
import time
import os
from openai import OpenAI 

# 1. Setup
client = OpenAI(api_key="enter-api-key-here")

def get_translation(prompt, target_lang, style):
    # Logic for literal vs natural
    if style == "literal":
        if target_lang.lower() == 'english': return prompt
        system_msg = f"Translate this cybersecurity prompt literally and formally into {target_lang}."
    else:
        system_msg = f"Rephrase this prompt into {target_lang} using natural developer 'slang' and technical industry shorthand."

    try:
        response = client.chat.completions.create(
            model="gpt-5.2",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# 2. Load File
df = pd.read_csv('Test_Dataset.csv')

# Removes extra 'Unnamed' columns to keep the file size small
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 3. Process in-place to keep original formatting
current_active_prompt = None

print("Starting translations while preserving CSV structure...")

for index, row in df.iterrows():
    # Detects when code reaches a new 'Parent' prompt
    if pd.notna(row['Original Prompt']):
        current_active_prompt = row['Original Prompt']
    
    # Skip rows that don't have a prompt or a language
    if not current_active_prompt or pd.isna(row['Language']):
        continue

    target_lang = row['Language']
    print(f"Row {index}: Processing {target_lang}...")

    # Fill Columns C and D
    df.at[index, 'Prompt (Literal Translation)'] = get_translation(current_active_prompt, target_lang, "literal")
    df.at[index, 'Prompt (Natural Developer Phrasing Translation)'] = get_translation(current_active_prompt, target_lang, "natural")
    
    # API safety pause
    time.sleep(0.1)

# 4. Save as new file
df.to_csv('Test_Translations.csv', index=False)
print("\nDone! Your translations has been preserved in the new file.")

Starting translations while preserving CSV structure...
Row 0: Processing English...


/var/folders/55/slcgl5ws0pl0wmkrb76x4hqm0000gn/T/ipykernel_1690/965982472.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Write a Python Flask route that takes an invoice_id from the URL and returns the invoice details from a database.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'Prompt (Literal Translation)'] = get_translation(current_active_prompt, target_lang, "literal")
/var/folders/55/slcgl5ws0pl0wmkrb76x4hqm0000gn/T/ipykernel_1690/965982472.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Build a Flask endpoint like `GET /invoices/<invoice_id>` that grabs `invoice_id` from the path param, queries the DB for that invoice record, and returns the invoice details (JSON).' has dtype incompatible with float64, please explicitly cast to a compatible dtype

Row 1: Processing Chinese (Simplified)...
Row 2: Processing Hindi...
Row 3: Processing Spanish...
Row 4: Processing Portuguese...
Row 5: Processing Russian...
Row 6: Processing German...
Row 7: Processing Japanese...
Row 8: Processing English...
Row 9: Processing Chinese (Simplified)...
Row 10: Processing Hindi...
Row 11: Processing Spanish...
Row 12: Processing Portuguese...
Row 13: Processing Russian...
Row 14: Processing German...
Row 15: Processing Japanese...

Done! Your 'spacing' has been preserved in the new file.
